# Trabalho Prático 1 
#### Leonardo Demore - 15674786
#### Arthur Araujo - 14651458

# Carregamento dos dados

In [1]:
import ir_datasets
import pandas as pd
from pathlib import Path

dataset = ir_datasets.load("cranfield")

docs = pd.DataFrame(
    dataset.docs_iter(),
    columns=["doc_id", "title", "text", "author", "bib"]
)

docs.head()

,doc_id,title,text,author,bib
0,1,experimental investigation of the aerodynamics...,experimental investigation of the aerodynamics...,"brenckman,m.","j. ae. scs. 25, 1958, 324."
1,2,simple shear flow past a flat plate in an inco...,simple shear flow past a flat plate in an inco...,ting-yili,"department of aeronautical engineering, rensse..."
2,3,the boundary layer in simple shear flow past a...,the boundary layer in simple shear flow past a...,m. b. glauert,"department of mathematics, university of manch..."
3,4,approximate solutions of the incompressible la...,approximate solutions of the incompressible la...,"yen,k.t.","j. ae. scs. 22, 1955, 728."
4,5,one-dimensional transient heat conduction into...,one-dimensional transient heat conduction into...,"wasserman,b.","j. ae. scs. 24, 1957, 924."


In [2]:
queries = pd.DataFrame(
    dataset.queries_iter(),
    columns=["query_id", "text"]
)

queries.head()

,query_id,text
0,1,what similarity laws must be obeyed when const...
1,2,what are the structural and aeroelastic proble...
2,3,what problems of heat conduction in composite ...
3,4,can a criterion be developed to show empirical...
4,5,what chemical kinetic system is applicable to ...


In [3]:
qrels = pd.DataFrame(
    dataset.qrels_iter(),
    columns=["query_id", "doc_id", "relevance", "iteration"]
)

qrels.head()

,query_id,doc_id,relevance,iteration
0,1,184,2,0
1,1,29,2,0
2,1,31,2,0
3,1,12,3,0
4,1,51,3,0


In [4]:
Path("data").mkdir(parents=True, exist_ok=True)

docs.to_csv("data/cranfield_docs.csv", index=False)
queries.to_csv("data/cranfield_queries.csv", index=False)
qrels.to_csv("data/cranfield_qrels.csv", index=False)

# Pré-processamento - Requisito 1


## Decisões de projeto

**Campos indexados.** Seguindo a convenção para esta coleção, indexamos **apenas `text`**: `author` e `bib` são metadados bibliográficos (nomes próprios, nomes de revistas, volumes, anos) que não descrevem o assunto do documento e só introduziriam ruído no casamento com as consultas.

**Tokenização por regex.** Usamos `re.findall(r"[a-z0-9]+", texto)`
em vez do `word_tokenize` do NLTK. A Cranfield é texto científico em inglês onde a pontuação é sempre separada por espaço (`slipstream .`).
A regex resolve normalização e tokenização em uma única passada. Mantemos dígitos porque em alguns documentos eles carregam sentido.

**Normalização para minúsculas.** Aplicada sempre em documentos e consultas, de modo que `Boundary` e `boundary` sejam o mesmo termo. Essa é a única normalização presente na configuração baseline.

**Stemming de Porter.** Reduz variantes morfológicas ao mesmo radical (ex: `aerodynamics` → `aerodynam`), aumentando o *recall*. Usamos o `PorterStemmer` do NLTK, permitido pela especificação.


### As quatro configurações comparadas

Conforme exigido pelo enunciado, avaliamos estas quatro combinações:

| Configuração | Stopwords | Stemming |
|---|---|---|
| `nada` | — | — |
| `stopwords` | removidas | — |
| `stemming` | — | Porter |
| `stopwords+stemming` | removidas | Porter |

### Leitura dos dados

In [5]:
import re
import time
import pandas as pd
from nltk.corpus import stopwords
from collections import Counter
from nltk.stem import PorterStemmer

DATA_DIR = "data"

# dtype=str nos identificadores para que doc_id/query_id casem entre os três
# arquivos sem depender do pandas. fillna("") porque dois
# documentos da coleção têm o campo de texto vazio.
docs = pd.read_csv(f"{DATA_DIR}/cranfield_docs.csv", dtype={"doc_id": str}).fillna("")
queries = pd.read_csv(f"{DATA_DIR}/cranfield_queries.csv", dtype={"query_id": str}).fillna("")
qrels = pd.read_csv(
    f"{DATA_DIR}/cranfield_qrels.csv", dtype={"query_id": str, "doc_id": str}
)

print(f"documentos: {len(docs)}")
print(f"consultas:  {len(queries)}")
print(f"qrels:      {len(qrels)}")

documentos: 1400
consultas:  225
qrels:      1837


### Função de pré-processamento

O pipeline é uma função única, `preprocess`, controlada por dois parâmetros
booleanos. Isso deixa as configurações do enunciado como quatro chamadas da mesma função.

O stemming é uma etapa cara, então os resultados são salvos em um
dicionário: cada termo distinto do vocabulário é reduzido uma única vez, embora
apareça muitas vezes na coleção.

In [6]:
TOKEN_RE = re.compile(r"[a-z0-9]+")
STOPWORDS = frozenset(stopwords.words("english"))
_stemmer = PorterStemmer()
_stem_cache = {}


def tokenize(text):
    """Normaliza para minúsculas e extrai sequências de letras e números"""
    return TOKEN_RE.findall(text.lower())


def stem(token):
    """Radical de Porter, com memoização por termo."""
    radical = _stem_cache.get(token)
    if radical is None:
        radical = _stemmer.stem(token)
        _stem_cache[token] = radical
    return radical


def preprocess(text, remove_stopwords=False, apply_stemming=False):
    """ 
    Ordem sempre fixa: tokenização, remoção de stopwords e por último stemming.
    É possível modificar a função pelos parâmetros booleanos.
    """
    tokens = tokenize(text)
    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS]
    if apply_stemming:
        tokens = [stem(t) for t in tokens]
    return tokens


print(f"stopwords do NLTK (inglês): {len(STOPWORDS)} termos")
print(f"amostra: {sorted(STOPWORDS)[:5]}")

stopwords do NLTK (inglês): 198 termos
amostra: ['a', 'about', 'above', 'after', 'again']


### Aplicação das quatro configurações

Guardamos o resultado em dois dicionários indexados pelo nome da configuração:
`doc_tokens[config]` e `query_tokens[config]` são listas de listas de termos, na
mesma ordem de `docs` e `queries`. Toda a coleção pré-processada cabe em memória com certa tranquilidade.

In [7]:
CONFIGS = {
    "nada": dict(remove_stopwords=False, apply_stemming=False),
    "stopwords": dict(remove_stopwords=True, apply_stemming=False),
    "stemming": dict(remove_stopwords=False, apply_stemming=True),
    "stopwords+stemming": dict(remove_stopwords=True, apply_stemming=True),
}

doc_tokens = {}
query_tokens = {}

for name, options in CONFIGS.items():
    doc_tokens[name] = [preprocess(t, **options) for t in docs["text"]]
    query_tokens[name] = [preprocess(t, **options) for t in queries["text"]]

# Índices auxiliares: posição na lista <-> identificador da coleção.
doc_ids = docs["doc_id"].tolist()
query_ids = queries["query_id"].tolist()

In [8]:
lines = []
for name in CONFIGS:
    vocabulary = set()
    terms_count = 0
    for terms in doc_tokens[name]:
        vocabulary.update(terms)
        terms_count += len(terms)
    query_count = sum(len(t) for t in query_tokens[name])

    lines.append(
        {
            "configuração": name,
            "vocabulário": len(vocabulary),
            "termos (total)": terms_count,
            "termos/doc": round(terms_count / len(docs), 1),
            "termos/consulta": round(query_count / len(queries), 1),
        }
    )

stats = pd.DataFrame(lines)
stats

,configuração,vocabulário,termos (total),termos/doc,termos/consulta
0,nada,7471,226526,161.8,17.4
1,stopwords,7352,132721,94.8,10.3
2,stemming,4820,226526,161.8,17.4
3,stopwords+stemming,4709,132721,94.8,10.3


### Efeito de cada configuração sobre a coleção

A tabela acima responde o requisito 1. Ela mede o que cada etapa faz, antes de qualquer recuperação.

As duas etapas atuam em **eixos diferentes**:

- **Remover stopwords** corta cerca de 41% do total de termos (226.526 → 132.721)
  mas apenas 119 entradas do vocabulário (7.471 → 7.352). São pouquíssimas
  palavras distintas, repetidas muitas vezes. O maior ganho é de **eficiência** (índice menor, menos posições a percorrer) e de redução de ruído na similaridade.

- **O stemming** não muda o total de termos (cada token continua
  existindo, mas com outra grafia), e reduz o vocabulário em 35% (7.471 →
  4.820), pois funde variantes da mesma palavra em um único termo. O ganho esperado
  é de **recall**.

Combinadas, as duas etapas produzem o vocabulário mais enxuto (4.709 termos) com
o menor número de termos indexados. Vale notar que a redução de vocabulário do
stemming é quase idêntica com e sem stopwords (35% contra 36%): as stopwords
contribuíam pouco para o vocabulário, então há pouca sobreposição entre o que
cada etapa elimina (as etapas são complementares).

### Resultado das configurações em um documento e uma consulta

In [9]:
print("DOCUMENTO 1 — primeiros 12 termos\n")
for name in CONFIGS:
    print(f"  {name:20s} {doc_tokens[name][0][:12]}")

print(f"\n\nCONSULTA 1 — {queries['text'][0].strip()!r}\n")
for name in CONFIGS:
    print(f"  {name:20s} {query_tokens[name][0]}")

DOCUMENTO 1 — primeiros 12 termos

  nada                 ['experimental', 'investigation', 'of', 'the', 'aerodynamics', 'of', 'a', 'wing', 'in', 'a', 'slipstream', 'an']
  stopwords            ['experimental', 'investigation', 'aerodynamics', 'wing', 'slipstream', 'experimental', 'study', 'wing', 'propeller', 'slipstream', 'made', 'order']
  stemming             ['experiment', 'investig', 'of', 'the', 'aerodynam', 'of', 'a', 'wing', 'in', 'a', 'slipstream', 'an']
  stopwords+stemming   ['experiment', 'investig', 'aerodynam', 'wing', 'slipstream', 'experiment', 'studi', 'wing', 'propel', 'slipstream', 'made', 'order']


CONSULTA 1 — 'what similarity laws must be obeyed when constructing aeroelastic models\nof heated high speed aircraft .'

  nada                 ['what', 'similarity', 'laws', 'must', 'be', 'obeyed', 'when', 'constructing', 'aeroelastic', 'models', 'of', 'heated', 'high', 'speed', 'aircraft']
  stopwords            ['similarity', 'laws', 'must', 'obeyed', 'constructing'

### Termos mais frequentes: mostra o resultado prático da etapa de stopwords

In [10]:
for nome in ("nada", "stopwords+stemming"):
    contagem = Counter()
    for termos in doc_tokens[nome]:
        contagem.update(termos)
    print(f"{nome}:")
    for termo, n in contagem.most_common(10):
        print(f"    {termo:14s} {n:6d}")
    print()

nada:
    the             19433
    of              12664
    and              6147
    a                5907
    in               4637
    to               4561
    is               4111
    for              3483
    are              2426
    with             2262

stopwords+stemming:
    flow             2080
    pressur          1390
    number           1345
    boundari         1214
    layer            1161
    result           1087
    effect            997
    method            888
    theori            883
    bodi              854



## Conclusão do requisito 1

Sem pré-processamento, os termos mais frequentes da coleção são palavras funcionais (*the*, *of*, *and*, *a*, ...). Com stopwords removidas e stemming aplicado, o topo da distribuição passa a ser o vocabulário técnico do domínio: `flow`, `pressur`, `number`, `boundari`, `layer` — condizente com uma coleção de aerodinâmica.

Isso antecipa um ponto importante para a Seção 2: a ponderação **TF-IDF** já
penaliza termos que ocorrem em quase todos os documentos, via o fator IDF. Ou
seja, parte do trabalho da remoção de stopwords é feito automaticamente pelo
peso dos termos. Por isso é razoável esperar que o ganho de *efetividade* da
remoção de stopwords no Modelo Vetorial seja modesto — o ganho maior é de
eficiência. O BM25, que também tem um componente de IDF, deve se comportar de
forma parecida.

# Modelo Vetorial - Requisito 2

## Decisões de projeto

**Peso dos termos.** Esquema `ltc`, aplicado tanto a documentos quanto a consultas:

$$w_{t,d} = (1 + \log_{10} \mathrm{tf}_{t,d}) \cdot \log_{10}\frac{N}{\mathrm{df}_t}$$

**Similaridade.** Cosseno entre os vetores de consulta e documento:

$$\mathrm{sim}(q,d) = \frac{\vec{q} \cdot \vec{d}}{\lVert \vec{q} \rVert \, \lVert \vec{d} \rVert}$$

$\lVert \vec{q} \rVert$ é constante para uma dada consulta e não altera a ordem do ranking, mas é aplicada para manter o score em $[0,1]$.

**Estrutura de dados.** O índice invertido mapeia termo → {documento: peso}, então o score percorre apenas os termos da consulta: documentos sem termo em comum nunca são visitados.

**Consultas.** Termos da consulta ausentes do índice são descartados (idf indefinido). O desempate é pelo índice do documento, para que o ranking seja reprodutível quando há scores iguais.

**Tamanho do ranking.** Guardamos o ranking completo (todos os documentos com score não nulo), e não só o Top-10, porque o MAP do requisito 4 precisa das posições de todos os relevantes.

### Índice invertido e ponderação

In [11]:
import math
from collections import defaultdict
from typing import NamedTuple


class VSM(NamedTuple):
    weights: dict  # termo -> {doc_idx: peso tf-idf}
    idf: dict      # termo -> idf
    norms: list     # ||d|| por documento


def build_inverted_index(tokens_per_doc):
    """termo -> {doc_idx: frequência do termo no documento}"""
    index = defaultdict(dict)
    for doc_idx, terms in enumerate(tokens_per_doc):
        for term, tf in Counter(terms).items():
            index[term][doc_idx] = tf
    return index


def build_vsm(tokens_per_doc):
    """Índice ponderado por tf-idf, com a norma de cada documento."""
    n_docs = len(tokens_per_doc)
    tf_index = build_inverted_index(tokens_per_doc)

    # df_t é o tamanho da lista de postings do termo.
    idf = {term: math.log10(n_docs / len(postings)) for term, postings in tf_index.items()}

    weights = {}
    norms = [0.0] * n_docs
    for term, postings in tf_index.items():
        weighted = {}
        for doc_idx, tf in postings.items():
            w = (1 + math.log10(tf)) * idf[term]
            weighted[doc_idx] = w
            norms[doc_idx] += w * w
        weights[term] = weighted

    return VSM(weights, idf, [math.sqrt(n) for n in norms])

### Busca

In [12]:
def vsm_search(model, query_terms, top_k=None):
    """Ranking [(doc_idx, score)] por cosseno, em ordem decrescente de score."""
    q_weights = {
        term: (1 + math.log10(tf)) * model.idf[term]
        for term, tf in Counter(query_terms).items()
        if term in model.idf
    }
    q_norm = math.sqrt(sum(w * w for w in q_weights.values()))
    if q_norm == 0:
        return []

    scores = defaultdict(float)
    for term, qw in q_weights.items():
        for doc_idx, dw in model.weights[term].items():
            scores[doc_idx] += qw * dw

    # norms == 0 nos dois documentos de texto vazio.
    ranking = [
        (doc_idx, score / (model.norms[doc_idx] * q_norm))
        for doc_idx, score in scores.items()
        if model.norms[doc_idx] > 0
    ]
    ranking.sort(key=lambda item: (-item[1], item[0]))
    return ranking[:top_k] if top_k else ranking

### Construção dos quatro modelos

In [13]:
vsm_models = {name: build_vsm(doc_tokens[name]) for name in CONFIGS}

vsm_rankings = {}
for name in CONFIGS:
    start = time.perf_counter()
    vsm_rankings[name] = {
        query_ids[i]: vsm_search(vsm_models[name], query_tokens[name][i])
        for i in range(len(query_ids))
    }
    elapsed = time.perf_counter() - start
    candidates = sum(len(r) for r in vsm_rankings[name].values()) / len(query_ids)
    print(f"{name:20s} {elapsed:5.2f}s   candidatos/consulta: {candidates:6.1f}")

nada                  0.17s   candidatos/consulta: 1366.3
stopwords             0.07s   candidatos/consulta:  730.8


stemming              0.20s   candidatos/consulta: 1373.3
stopwords+stemming    0.09s   candidatos/consulta:  904.6


### Verificação da implementação

Um documento usado como consulta contra si mesmo deve ter cosseno exatamente 1,0 — é o teste mais direto de que a ponderação e a normalização estão corretas.

In [14]:
check = vsm_search(vsm_models["stopwords+stemming"], doc_tokens["stopwords+stemming"][0], top_k=3)
print("documento 1 como consulta:")
for doc_idx, score in check:
    print(f"    doc {doc_ids[doc_idx]:>4s}  {score:.4f}")

all_scores = [s for r in vsm_rankings["stopwords+stemming"].values() for _, s in r]
print(f"\nscores: min={min(all_scores):.6f}  max={max(all_scores):.6f}")

documento 1 como consulta:
    doc    1  1.0000
    doc  484  0.2683
    doc 1064  0.2157

scores: min=0.000605  max=0.753408


### Exemplo de ranking

Marcamos com `R` os documentos relevantes segundo o qrels (grau ≥ 1). A marcação é apenas ilustrativa — os julgamentos não participam da geração do ranking.

In [15]:
relevant = defaultdict(set)
for row in qrels.itertuples():
    if row.relevance >= 1:
        relevant[row.query_id].add(row.doc_id)

query_text = dict(zip(query_ids, queries["text"]))


def show_ranking(rankings, query_id, k=10):
    print(f"CONSULTA {query_id} — {' '.join(query_text[query_id].split())}\n")
    for position, (doc_idx, score) in enumerate(rankings[query_id][:k], 1):
        doc_id = doc_ids[doc_idx]
        mark = "R" if doc_id in relevant[query_id] else " "
        title = " ".join(docs["title"][doc_idx].split())[:58]
        print(f"  {position:2d}. [{mark}] doc {doc_id:>4s}  {score:.4f}  {title}")


show_ranking(vsm_rankings["stopwords+stemming"], "1")

CONSULTA 1 — what similarity laws must be obeyed when constructing aeroelastic models of heated high speed aircraft .

   1. [ ] doc  573  0.2346  viscous hypersonic similitude .
   2. [ ] doc  944  0.2065  one dimensional heat conduction through the skin of a vehi
   3. [R] doc   51  0.1988  theory of aircraft structural models subjected to aerodyna
   4. [R] doc  184  0.1768  scale models for thermo-aeroelastic research .
   5. [ ] doc  878  0.1629  experimental model techniques and equipment for flutter in
   6. [ ] doc  486  0.1612  similarity laws for aerothermoelastic testing .
   7. [R] doc   12  0.1558  some structural and aerelastic considerations of high spee
   8. [ ] doc  665  0.1513  on the theory of hypersonic gas flow with a power law shoc
   9. [R] doc  879  0.1400  flutter model testing at transonic speeds .
  10. [ ] doc 1361  0.1281  large deflections of structures subjected to heating and e


## Conclusão do requisito 2

O cosseno de um documento contra si mesmo é exatamente 1,0 e nenhum score sai de $[0,1]$, o que valida a ponderação e a normalização.

O tamanho do conjunto de candidatos mostra o efeito prático das stopwords na recuperação:

| configuração | candidatos/consulta |
|---|---|
| `nada` | 1366,3 |
| `stopwords` | 730,8 |
| `stemming` | 1373,3 |
| `stopwords+stemming` | 904,6 |

Sem remoção de stopwords, **1366 dos 1400 documentos** recebem score não nulo: basta um *the* em comum para o documento entrar no cálculo. O ranking final não muda muito, porque o idf desses termos é próximo de zero, mas percorremos quase toda a coleção para chegar nele. Com stopwords removidas o conjunto cai para 731 documentos, e o stemming o amplia de novo (904,6) — coerente com o que ele faz: fundir variantes aumenta o número de documentos que casam com algum termo da consulta, que é justamente o ganho de recall esperado.

No Top-10 da consulta 1, quatro dos dez documentos são relevantes. Vale registrar um caso para a análise de erros do requisito 9: o documento 486, *similarity laws for aerothermoelastic testing*, aparece na 6ª posição com título quase idêntico ao tema da consulta (*what similarity laws must be obeyed...*), mas **não** consta no qrels como relevante — indício de julgamento incompleto na coleção, e não de falha do modelo.